In [1]:
import json
import re
import time
import gc
from pathlib import Path

import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GB


In [4]:
TEST_PATH = Path("data/splits/test.jsonl")

assert TEST_PATH.exists(), f"Test dataset not found: {TEST_PATH}"

with TEST_PATH.open("r", encoding="utf-8") as f:
    test_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print(f"Test examples: {len(test_records):,}")

Test examples: 2,929


In [5]:
example = test_records[0]

print("QUERY:")
print(example["query"])

print("\nTOOLS:")
print(json.dumps(example["tools"], indent=2))

print("\nGOLD ANSWER:")
print(json.dumps(example["answers"], indent=2))

QUERY:
Fetch details for product 456789 with locale 'es_ES'.

TOOLS:
[
  {
    "name": "get_product",
    "description": "Fetches product details from an API using the given product ID.",
    "parameters": {
      "is_id": {
        "description": "The product ID for which to retrieve details.",
        "type": "int",
        "default": ""
      }
    }
  },
  {
    "name": "product_id",
    "description": "Fetches product details from the AliExpress API using the provided product ID and optional parameters.",
    "parameters": {
      "is_id": {
        "description": "The ID of the product to fetch details for.",
        "type": "int",
        "default": "4000886597329"
      },
      "locale": {
        "description": "The locale to be used for the request (e.g., 'en_US').",
        "type": "str, optional",
        "default": ""
      },
      "currency": {
        "description": "The currency for price conversion (e.g., 'USD').",
        "type": "str, optional",
        "default": 

In [6]:
SYSTEM_PROMPT = """You are a function-calling assistant.

Your task is to select the appropriate tool and provide the arguments required
to answer the user's request.

Return ONLY valid JSON.

The output must be a JSON array of tool calls in this format:

[
  {
    "name": "function_name",
    "arguments": {
      "argument_name": "value"
    }
  }
]

Do not explain your answer.
Do not output markdown.
Do not invent tools.
"""

In [7]:
def build_user_prompt(record):
    tools_json = json.dumps(
        record["tools"],
        ensure_ascii=False,
        indent=2
    )

    return f"""Available tools:

{tools_json}

User request:

{record["query"]}

Return the required tool call(s) as JSON."""

In [8]:
print(build_user_prompt(test_records[0]))

Available tools:

[
  {
    "name": "get_product",
    "description": "Fetches product details from an API using the given product ID.",
    "parameters": {
      "is_id": {
        "description": "The product ID for which to retrieve details.",
        "type": "int",
        "default": ""
      }
    }
  },
  {
    "name": "product_id",
    "description": "Fetches product details from the AliExpress API using the provided product ID and optional parameters.",
    "parameters": {
      "is_id": {
        "description": "The ID of the product to fetch details for.",
        "type": "int",
        "default": "4000886597329"
      },
      "locale": {
        "description": "The locale to be used for the request (e.g., 'en_US').",
        "type": "str, optional",
        "default": ""
      },
      "currency": {
        "description": "The currency for price conversion (e.g., 'USD').",
        "type": "str, optional",
        "default": ""
      },
      "country": {
        "description

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Model loaded.")

W1003 19:48:36.599000 6284 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\Swapn\miniconda3\envs\unsloth\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Swapn\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [10]:
def generate_response(model, tokenizer, record):
    
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": build_user_prompt(record)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():

        start = time.perf_counter()

        output_ids = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

        elapsed = time.perf_counter() - start

    generated_ids = output_ids[0][inputs.shape[-1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response, elapsed

In [11]:
response, latency = generate_response(
    model,
    tokenizer,
    test_records[0]
)

print(response)
print(f"\nLatency: {latency:.3f}s")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AttributeError: 

In [ ]:
def extract_json(text):
    
    text = text.strip()

    # Remove markdown fences
    text = re.sub(
        r"```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace("```", "").strip()

    # Try complete response first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting JSON array
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1 and end > start:

        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass

    # Try extracting JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:

        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass

    return None

In [ ]:
parsed = extract_json(response)

print(parsed)
print(type(parsed))

In [ ]:
def normalize_calls(calls):

    if calls is None:
        return None

    if isinstance(calls, dict):

        # Some models may output a single call object
        if "name" in calls:
            calls = [calls]
        else:
            return None

    if not isinstance(calls, list):
        return None

    normalized = []

    for call in calls:

        if not isinstance(call, dict):
            return None

        name = call.get("name")

        arguments = call.get(
            "arguments",
            {}
        )

        if not isinstance(name, str):
            return None

        if not isinstance(arguments, dict):
            return None

        normalized.append({
            "name": name,
            "arguments": arguments
        })

    return normalized

In [ ]:
def canonicalize_calls(calls):

    if calls is None:
        return None

    return [
        {
            "name": call["name"],
            "arguments": call["arguments"]
        }
        for call in calls
    ]

In [ ]:
def evaluate_prediction(predicted, gold, tools):

    result = {
        "parse_valid": 0,
        "valid_call": 0,
        "tool_em": 0,
        "function_accuracy": 0,
        "argument_accuracy": 0,
    }

    predicted = normalize_calls(predicted)
    gold = normalize_calls(gold)

    if predicted is None:
        return result

    result["parse_valid"] = 1

    if gold is None:
        return result

    # Tool names available to model
    available_tools = {
        tool.get("name")
        for tool in tools
        if isinstance(tool, dict)
    }

    # Check structural validity
    valid = True

    for call in predicted:

        if call["name"] not in available_tools:
            valid = False
            break

    if valid:
        result["valid_call"] = 1

    # Exact match
    if predicted == gold:
        result["tool_em"] = 1

    # Function accuracy
    if len(predicted) == len(gold):

        function_matches = [
            p["name"] == g["name"]
            for p, g in zip(predicted, gold)
        ]

        if all(function_matches):
            result["function_accuracy"] = 1

            # Argument accuracy
            argument_matches = [
                p["arguments"] == g["arguments"]
                for p, g in zip(predicted, gold)
            ]

            if all(argument_matches):
                result["argument_accuracy"] = 1

    return result

In [ ]:
predicted = extract_json(response)

metrics = evaluate_prediction(
    predicted,
    gold_records[0]["answers"],
    gold_records[0]["tools"]
)

print(metrics)

In [ ]:
def benchmark_model(
    model,
    tokenizer,
    records,
    model_name,
    limit=None
):

    if limit is not None:
        records = records[:limit]

    results = []

    for i, record in enumerate(records):

        try:

            response, latency = generate_response(
                model,
                tokenizer,
                record
            )

            parsed = extract_json(response)

            metrics = evaluate_prediction(
                parsed,
                record["answers"],
                record["tools"]
            )

            results.append({
                "index": i,
                "model": model_name,
                "latency": latency,
                "raw_response": response,
                "prediction": parsed,
                **metrics
            })

        except Exception as e:

            results.append({
                "index": i,
                "model": model_name,
                "latency": None,
                "raw_response": None,
                "prediction": None,
                "parse_valid": 0,
                "valid_call": 0,
                "tool_em": 0,
                "function_accuracy": 0,
                "argument_accuracy": 0,
                "error": str(e)
            })

        if (i + 1) % 25 == 0:
            print(f"{i + 1}/{len(records)}")

    return pd.DataFrame(results)

In [ ]:
pilot_results = benchmark_model(
    model,
    tokenizer,
    gold_records,
    MODEL_NAME,
    limit=25
)

In [ ]:
pilot_results[
    [
        "parse_valid",
        "valid_call",
        "tool_em",
        "function_accuracy",
        "argument_accuracy",
        "latency"
    ]
].mean(numeric_only=True)

In [ ]:
MODEL_CANDIDATES = {
    "SmolLM2-360M": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen2.5-0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "Gemma3-270M": "google/gemma-3-270m-it",
    "LFM2-350M": "LiquidAI/LFM2-350M",
}

In [ ]:
all_results = []

for short_name, model_name in MODEL_CANDIDATES.items():

    print("\n" + "=" * 70)
    print(f"MODEL: {short_name}")
    print("=" * 70)

    try:

        tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )

        model.eval()

        results = benchmark_model(
            model,
            tokenizer,
            gold_records,
            short_name
        )

        all_results.append(results)

        # Free GPU memory
        del model
        del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:

        print(f"FAILED: {e}")

In [ ]:
benchmark_df = pd.concat(
    all_results,
    ignore_index=True
)

print(
    benchmark_df.shape
)

In [ ]:
Path("data/reports").mkdir(
    parents=True,
    exist_ok=True
)

benchmark_df.to_json(
    "data/reports/model_selection_raw.jsonl",
    orient="records",
    lines=True
)

In [ ]:
selection_table = (
    benchmark_df
    .groupby("model")
    .agg(
        tool_em=("tool_em", "mean"),
        function_accuracy=("function_accuracy", "mean"),
        argument_accuracy=("argument_accuracy", "mean"),
        valid_call=("valid_call", "mean"),
        parse_valid=("parse_valid", "mean"),
        latency_p50=("latency", "median"),
        latency_mean=("latency", "mean"),
    )
    .reset_index()
)

selection_table

In [ ]:
accuracy_columns = [
    "tool_em",
    "function_accuracy",
    "argument_accuracy",
    "valid_call",
    "parse_valid"
]

selection_table[accuracy_columns] *= 100

selection_table